In [ ]:
# Install the required libraries
!pip -q install google-genai pandas openpyxl

# Import libraries
from google import genai
from google.genai import types
from google.genai.errors import APIError
from google.colab import userdata, files
import pandas as pd
import time

# Connect to Gemini using the API key saved in Colab Secrets
# genai allows Python to communicate with Gemini.
client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY")
)

# Choosing our model
MODEL = "gemini-3.5-flash-lite"

print("Setup completed!")

### Optional Practice — Explore Available Gemini Models

This section is **optional** and is provided for practice and exploration.

The code below displays the Gemini models that are listed as supporting text generation through the API.

You can explore the available models to understand that Gemini offers different models for different tasks.

**Note:** Not every listed model is necessarily available under the free tier. Keep the configured model if it is available to your account, or choose another available text-generation model.

In [ ]:
# List models that support text generation
for model in client.models.list():
    if "generateContent" in (model.supported_actions or []):
        print(model.name)

### Optional Practice — Translate a Single Word

This section is **optional** and is intended to help you understand how a basic API call works.

Before translating multiple sentences from an Excel file, you can test the Gemini API by translating just **one Arabic word** into English.

In the example below, we use the Arabic word **"كتاب" (Book)**.

You can change this word to any other Arabic word and run the code again to see the result.

This example demonstrates how to:
- Write a simple prompt.
- Send a request to Gemini using the API.
- Receive and display the generated translation.

**Note:** This is an optional practice example.

In [ ]:
# The Arabic word we want to translate
word = "كتاب"

# Instructions for Gemini
prompt = f"Translate this Arabic word into English: {word}. Return only the translation."

# Send the request to Gemini
response = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config=types.GenerateContentConfig(temperature=0)
)

# Display the result
print("Arabic:", word)
print("English:", response.text)

In [ ]:
# Upload an Excel file from your computer
uploaded = files.upload()

# Get the uploaded file's name
filename = next(iter(uploaded))

# Read the Excel file
df = pd.read_excel(filename)

# Prepare the translation column to store text
df["English Translation (API)"] = df["English Translation (API)"].astype("string")

# Display information about our dataset
print("Total sentences:", len(df))
display(df.head())

In [ ]:
# Select only the first five sentences
test_df = df.head(5).copy()

# Go through the sentences one by one
for i, row in test_df.iterrows():

    # Read the Arabic sentence and its topic
    sentence = row["Sentence (Arabic)"]
    topic = row["Topic"]

    # Create the prompt
    prompt = f"""
    Translate this Arabic sentence into English.
    Topic: {topic}
    Sentence: {sentence}
    Return only the translation.
    """

    # Call Gemini
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0)
    )

    # Save the English translation
    test_df.loc[i, "English Translation (API)"] = response.text.strip()

    # Show progress
    print(f"Translated sentence {i + 1}")

    # Wait briefly between requests
    time.sleep(3)

# Display the translated sentences
display(test_df)

### Optional Practice — Translate the Entire Excel File

This section is **optional** and is provided as additional information for students who would like to explore further.

In the previous example, we used `df.head(5)` to translate **only the first five sentences** from the Excel file.

The code below uses `df.iterrows()` instead, which allows Python to go through **every row in the Excel file** and translate all sentences automatically.

It also includes additional features:
- Skips sentences that have already been translated.
- Automatically retries requests when the API rate limit is reached.
- Saves progress after every 10 rows.
- Downloads the translated Excel file when finished.

**Important:** The full Excel file contains 100 sentences, so running this code may require more API calls and additional waiting time due to free-tier limits.

You can start with a small set of sentences before translating the entire provided Excel file.

This section is simply included to demonstrate how the same code can be extended to process larger datasets.

In [ ]:
# Translate all sentences
for i, row in df.iterrows():

    # Skip sentences that already have translations
    if pd.notna(df.loc[i, "English Translation (API)"]):
        continue

    sentence = row["Sentence (Arabic)"]
    topic = row["Topic"]

    # Create the prompt
    prompt = f"""
    Translate this Arabic sentence into English.
    Topic: {topic}
    Sentence: {sentence}
    Return only the translation.
    """

    # Try each sentence up to 3 times
    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model=MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(temperature=0)
            )

            if response.text:
                df.loc[i, "English Translation (API)"] = response.text.strip()
                print(f"Translated {i + 1}/{len(df)}")

            break  # Success: stop retrying this sentence

        except APIError as e:
            if e.code == 429 and attempt < 2:
                print("Rate limit reached. Waiting 65 seconds...")
                time.sleep(65)
            else:
                print(f"API error at sentence {i + 1}: {e}")
                break

    # Save progress after every 10 rows
    if (i + 1) % 10 == 0:
        df.to_excel("Translation_Progress.xlsx", index=False)
        print("Progress saved.")

    # Wait between requests
    time.sleep(6)

# Save the completed or partially completed file
df.to_excel("Translated_Sentences.xlsx", index=False)

print(
    "Completed:",
    df["English Translation (API)"].notna().sum(),
    "/",
    len(df)
)

files.download("Translated_Sentences.xlsx")